# SNAPCHAT SPOTLIGHT ALGORITHMIC ENGAGEMENT & RETENTION OPTIMIZATION
## Master Reproducible Portfolio Notebook

**Target:** Growth Data Scientist, Level 4 — `Q326DSA4`
**Project:** `driiiportfolio`
**Dataset:** `snap_spotlight_growth`
**Repository:** `snap-spotlight-growth-ds-l4`

This notebook is the clean execution path. It consolidates the project scenario, data contract, synthetic generation, BigQuery ingestion, validation, statistical analysis, and rollout evaluation. Supporting SQL and documentation files remain the source-controlled artifacts for GitHub.

## Business Scenario

A hypothetical product team is evaluating an Algorithmic Exploration recommendation treatment for Snapchat Spotlight against Standard Recommendations. The treatment may increase short-term sharing and discovery, while creating possible longer-term retention or user-experience tradeoffs. This is a synthetic portfolio scenario aligned to the supplied Snap Data Scientist Level 4 job description and is not a claim about an actual internal Snap incident.

## Business Problem / Challenge / Bottleneck

The project must determine whether a recommendation treatment changes engagement, exact-day retention, and guardrail metrics while preserving experiment integrity. The objective is to demonstrate a decision system that can detect both positive and negative product effects before broad rollout.

## Methodology

1. Generate deterministic synthetic data.
2. Enforce an explicit BigQuery schema.
3. Validate data quality and experiment assignment.
4. Build user-level retention and guardrail metrics.
5. Test SRM.
6. Estimate treatment effects with uncertainty.
7. Run logistic regression as an associational model.
8. Evaluate staged rollout decisions.
9. Publish dashboard-facing views.

## Part 1 — Data Flow Architecture & System Relationships

`Python/Colab → CSV artifacts → BigQuery raw tables → SQL analytical views → Colab statistical analysis → Looker Studio → Executive decision`.

The experiment unit is the user. Sessions are event-level observations and are aggregated to the user level before user-level inference.

## Part 2 — GitHub Repository Documentation

The repository includes `README.md`, `Executive_Summary.md`, `Project_Disclaimer.md`, `Dashboard_Executive_Summary.md`, `PROCESS_MANUAL.md`, and the `docs/` specifications.

## Part 3 — Reproducible Code Codebase

SQL is stored under `sql/`; executable notebooks under `notebooks/`; validation queries under `tests/`. The current notebook is the master execution path.

## Part 4 — Phase-by-Phase Process Manual

Follow: environment → generation → schema → load → validation → SQL views → statistical analysis → staged rollout → dashboard → executive documentation. See `PROCESS_MANUAL.md` for the detailed order of operations and error handling.

## Phase 0 — Configuration

In [ ]:
import os
import numpy as np
import pandas as pd
from datetime import datetime, timedelta

RANDOM_SEED = 42
PROJECT_ID = "driiiportfolio"
DATASET_ID = "snap_spotlight_growth"
NUM_USERS = 25_000
NUM_SESSIONS = 150_000
START_DATE = datetime(2026, 8, 1)

np.random.seed(RANDOM_SEED)


## Treatment Parameters

The parameters below are explicit synthetic assumptions. `d28_treatment_effect` is implemented in the Day-28 return probability.

## Phase 1 — Synthetic Data Generation

In [ ]:
SHARES_TREATMENT_EFFECT = 0.35
D28_TREATMENT_EFFECT = -0.025
DISSATISFACTION_BASE_RATE = 0.03
VARIANT_DISSATISFACTION_MULT = 1.85
CONTROL_D28_RATE = 0.80
CONTROL_D7_RATE = 0.70
VARIANT_D7_DELTA = 0.00

assert -1 < D28_TREATMENT_EFFECT < 1


## Generate Users

In [ ]:
user_ids = [f"usr_{i:06d}" for i in range(1, NUM_USERS + 1)]
cohort_dates = [
    (START_DATE + timedelta(days=int(np.random.exponential(scale=7)))).date()
    for _ in range(NUM_USERS)
]
device_os = np.random.choice(["iOS", "Android"], NUM_USERS, p=[0.58, 0.42])
region = np.random.choice(["US_WEST", "US_EAST", "EU_WEST", "APAC"], NUM_USERS, p=[0.40, 0.30, 0.18, 0.12])
experiment_group = np.random.choice(
    ["Control_Standard_Recs", "Variant_Algorithmic_Exploration"],
    NUM_USERS, p=[0.50, 0.50]
)

dim_users = pd.DataFrame({
    "user_id": user_ids,
    "cohort_date": cohort_dates,
    "device_os": device_os,
    "region": region,
    "experiment_group": experiment_group
})

dim_users.head()


## Generate Session Events

The generator first creates a guaranteed onboarding session for every user, then samples additional sessions until the requested target is reached. Day-specific return events are generated independently so exact-day retention is measurable.

In [ ]:
rng = np.random.default_rng(RANDOM_SEED)
user_index = dim_users.set_index("user_id")

records = []

def add_session(uid, day_offset, group, is_retention_event=False):
    cohort = user_index.loc[uid, "cohort_date"]
    ts = pd.Timestamp(cohort) + pd.Timedelta(days=int(day_offset)) + pd.Timedelta(hours=float(rng.uniform(0, 24)))
    variant = group == "Variant_Algorithmic_Exploration"
    completion = rng.beta(2, 4.8 if variant else 5)
    latency = max(50, int(rng.normal(218 if variant else 210, 35)))
    lam = 0.85 * (1 + SHARES_TREATMENT_EFFECT) if variant else 0.85
    shares = int(rng.poisson(lam))
    neg_prob = DISSATISFACTION_BASE_RATE * (VARIANT_DISSATISFACTION_MULT if variant else 1.0)
    negative = int(rng.random() < neg_prob)
    records.append({
        "session_id": f"sess_{len(records)+1:08d}",
        "user_id": uid,
        "session_timestamp": ts,
        "avg_watch_completion": round(float(np.clip(completion, 0.05, 1.0)), 4),
        "avg_latency_ms": latency,
        "total_shares": shares,
        "negative_feedback_flag": negative
    })

# One onboarding session per user.
for uid, row in dim_users.set_index("user_id").iterrows():
    add_session(uid, 0, row["experiment_group"])

# Exact-day retention probabilities.
for _, row in dim_users.iterrows():
    uid = row.user_id
    group = row.experiment_group
    d7_p = CONTROL_D7_RATE + VARIANT_D7_DELTA if group == "Variant_Algorithmic_Exploration" else CONTROL_D7_RATE
    d28_p = CONTROL_D28_RATE + (D28_TREATMENT_EFFECT if group == "Variant_Algorithmic_Exploration" else 0)
    if rng.random() < d7_p:
        add_session(uid, 7, group, True)
    if rng.random() < d28_p:
        add_session(uid, 28, group, True)

# Fill remaining volume with ordinary sessions on Days 1–27.
while len(records) < NUM_SESSIONS:
    row = dim_users.iloc[int(rng.integers(0, len(dim_users)))]
    add_session(row.user_id, int(rng.integers(1, 28)), row.experiment_group)

fact_user_sessions = pd.DataFrame(records[:NUM_SESSIONS])
fact_user_sessions["session_timestamp"] = pd.to_datetime(fact_user_sessions["session_timestamp"])

print(dim_users.shape)
print(fact_user_sessions.shape)


## Local Validation

In [ ]:
assert len(dim_users) == NUM_USERS
assert len(fact_user_sessions) == NUM_SESSIONS
assert dim_users.user_id.is_unique
assert fact_user_sessions.session_id.is_unique
assert fact_user_sessions.user_id.isin(dim_users.user_id).all()
assert fact_user_sessions.avg_watch_completion.between(0, 1).all()
assert (fact_user_sessions.avg_latency_ms >= 0).all()
assert (fact_user_sessions.total_shares >= 0).all()
assert fact_user_sessions.negative_feedback_flag.isin([0,1]).all()
print("Validation passed.")


## Export CSV Artifacts

In [ ]:
os.makedirs("data", exist_ok=True)
dim_users.to_csv("data/dim_users.csv", index=False)
fact_user_sessions.to_csv("data/fact_user_sessions.csv", index=False)
print("Wrote data/dim_users.csv and data/fact_user_sessions.csv")


## BigQuery Upload — Explicit Schema

This cell is optional. It avoids the prior `REQUIRED → NULLABLE` failure by supplying the exact BigQuery schema and using `WRITE_TRUNCATE`.

In [ ]:
from google.cloud import bigquery

client = bigquery.Client(project=PROJECT_ID)

dim_schema = [
    bigquery.SchemaField("user_id", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("cohort_date", "DATE", mode="REQUIRED"),
    bigquery.SchemaField("device_os", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("region", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("experiment_group", "STRING", mode="REQUIRED"),
]

session_schema = [
    bigquery.SchemaField("session_id", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("user_id", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("session_timestamp", "TIMESTAMP", mode="REQUIRED"),
    bigquery.SchemaField("avg_watch_completion", "FLOAT64", mode="REQUIRED"),
    bigquery.SchemaField("avg_latency_ms", "INT64", mode="REQUIRED"),
    bigquery.SchemaField("total_shares", "INT64", mode="REQUIRED"),
    bigquery.SchemaField("negative_feedback_flag", "INT64", mode="REQUIRED"),
]

dim_users["cohort_date"] = pd.to_datetime(dim_users["cohort_date"]).dt.date

client.load_table_from_dataframe(
    dim_users,
    f"{PROJECT_ID}.{DATASET_ID}.dim_users",
    job_config=bigquery.LoadJobConfig(schema=dim_schema, write_disposition="WRITE_TRUNCATE")
).result()

client.load_table_from_dataframe(
    fact_user_sessions,
    f"{PROJECT_ID}.{DATASET_ID}.fact_user_sessions",
    job_config=bigquery.LoadJobConfig(schema=session_schema, write_disposition="WRITE_TRUNCATE")
).result()

print("BigQuery load completed with explicit schemas.")


## Phase 2 — Statistical / Causal Analysis

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import statsmodels.api as sm
from google.cloud import bigquery

PROJECT_ID = "driiiportfolio"
DATASET_ID = "snap_spotlight_growth"
client = bigquery.Client(project=PROJECT_ID)


In [ ]:
query = f"SELECT * FROM `{PROJECT_ID}.{DATASET_ID}.analysis_dataset`"
df = client.query(query).to_dataframe()
print(df.shape)
df.head()


## 1. Sample Ratio Mismatch

In [ ]:
counts = df["experiment_group"].value_counts().reindex(["Control_Standard_Recs", "Variant_Algorithmic_Exploration"])
expected = np.repeat(len(df) / 2, 2)
chi2, srm_p = stats.chisquare(counts.values, f_exp=expected)
print({"control": int(counts.iloc[0]), "variant": int(counts.iloc[1]), "chi2": chi2, "p_value": srm_p})


## 2. Retention Summary

In [ ]:
retention_cols = ["retained_d1", "retained_d3", "retained_d7", "retained_d28"]
summary = df.groupby("experiment_group")[retention_cols].mean()
summary


## 3. Binary D28 Effect

In [ ]:
c = df.loc[df.experiment_group == "Control_Standard_Recs", "retained_d28"]
v = df.loc[df.experiment_group == "Variant_Algorithmic_Exploration", "retained_d28"]

diff = v.mean() - c.mean()
relative = diff / c.mean()
# Normal approximation for a two-proportion confidence interval.
p1, p2 = v.mean(), c.mean()
se = np.sqrt(p1*(1-p1)/len(v) + p2*(1-p2)/len(c))
ci_low, ci_high = diff - 1.96*se, diff + 1.96*se
z = diff / se
p_value = 2 * stats.norm.sf(abs(z))

print({
    "control_rate": c.mean(),
    "variant_rate": v.mean(),
    "absolute_difference": diff,
    "relative_difference": relative,
    "ci_low": ci_low,
    "ci_high": ci_high,
    "p_value": p_value
})


## 4. Engagement Effects — Welch t-tests

In [ ]:
for metric in ["total_shares", "avg_watch_completion", "avg_latency_ms"]:
    control = df.loc[df.experiment_group == "Control_Standard_Recs", metric]
    variant = df.loc[df.experiment_group == "Variant_Algorithmic_Exploration", metric]
    t_stat, p_val = stats.ttest_ind(variant, control, equal_var=False)
    print(metric, {"control_mean": control.mean(), "variant_mean": variant.mean(), "difference": variant.mean()-control.mean(), "t": t_stat, "p": p_val})


## 5. Logistic Regression — D28 Retention Association

This model is descriptive/predictive. Its coefficients should not be presented as causal effects of the predictors.

In [ ]:
model_df = df.copy()
model_df["variant"] = (model_df.experiment_group == "Variant_Algorithmic_Exploration").astype(int)
features = ["variant", "retained_d7", "total_shares", "avg_watch_completion", "avg_latency_ms", "negative_feedback_flag"]
X = sm.add_constant(model_df[features].astype(float))
y = model_df["retained_d28"].astype(int)
model = sm.Logit(y, X).fit(disp=False)
print(model.summary())
print("Odds ratios:")
print(np.exp(model.params))


## 6. Staged Rollout Decision Engine

Stages use progressively larger user subsets. A production implementation would use time-based exposure cohorts and live monitoring; here the stage subsets are deterministic portfolio simulation cohorts.

In [ ]:
def evaluate_stage(stage_df, exposure_rate, d28_threshold=-0.01, negative_feedback_relative_threshold=0.50):
    c = stage_df.loc[stage_df.experiment_group == "Control_Standard_Recs"]
    v = stage_df.loc[stage_df.experiment_group == "Variant_Algorithmic_Exploration"]
    d28_diff = v.retained_d28.mean() - c.retained_d28.mean()
    c_nf, v_nf = c.negative_feedback_flag.mean(), v.negative_feedback_flag.mean()
    nf_rel = (v_nf / c_nf - 1) if c_nf > 0 else np.inf
    if d28_diff < d28_threshold or nf_rel > negative_feedback_relative_threshold:
        decision = "ROLLBACK"
    else:
        decision = "ADVANCE"
    return {"exposure_rate": exposure_rate, "users": len(stage_df), "d28_diff": d28_diff, "negative_feedback_relative_change": nf_rel, "decision": decision}

stages = [0.01, 0.05, 0.10, 1.00]
stage_results = []
for i, rate in enumerate(stages, 1):
    n = max(2, int(len(df) * rate))
    stage_df = df.sort_values("user_id").head(n)
    stage_results.append({"stage": i, **evaluate_stage(stage_df, rate)})

stage_results_df = pd.DataFrame(stage_results)
stage_results_df


## 7. Final Interpretation Template

Complete the following after execution. Do not invent results.

- **Experiment integrity:** [observed SRM result]
- **Engagement:** [observed treatment effect]
- **D1/D3/D7:** [observed leading indicators]
- **D28:** [observed primary outcome, effect, CI, p-value]
- **Guardrails:** [negative feedback / latency]
- **Predictive model:** [key associations with uncertainty]
- **Rollout state:** [ADVANCE / HOLD / ROLLBACK]
- **Next action:** [evidence-linked action]

## Final Executive Interpretation

Complete only after execution. Do not invent statistics.

- Experiment integrity: [observed]
- Engagement effect: [observed]
- D1/D3/D7: [observed]
- D28: [observed]
- Guardrails: [observed]
- Predictive associations: [observed]
- Rollout state: [ADVANCE / HOLD / ROLLBACK]
- Next action: [evidence-linked]